# Lab 03-03: Build a State-Based Returns Flow in Code

**Book chapter:** Chapter 03: Building Agents With Low-Code Tools  
**Estimated time:** 35 minutes

## Overview

Cymbal Retail's returns process must follow exact eligibility rules every time, so
the team does not want a generative model to decide eligibility. Instead, a
deterministic flow collects the facts it needs and applies the rules.

In this lab, you use the Dialogflow CX API to build a returns flow from an intent,
pages, a form, condition routes, and an event handler, and then you test it with
scripted conversations. Flows built this way are the same flows that CX Agent Studio
imports as flow-based agents beneath an instruction-based root agent.

## Learning Objectives

By the end of this lab, you will be able to:

- Create a flow-based agent and an intent with training phrases
- Build pages with entry fulfillment and a form that collects parameters
- Control transitions with intent routes, condition routes, and END_SESSION
- Recover from unexpected input with a sys.no-match-default event handler
- Test a flow with the DetectIntent API and inspect its state

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for the test conversations. Run the cleanup step at the end to delete the agent.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

A flow-based conversation behaves like a **state machine**: at any moment it is on
exactly one page, and routes and handlers decide which page comes next.

| Building block | Role in this lab |
|---|---|
| **Intent route** | Moves from the start page to the returns page when the customer asks to return an item |
| **Page and entry fulfillment** | A conversational state and the message sent when the state becomes active |
| **Form and parameters** | The values a page must collect, in order: the order number and the days since delivery |
| **Condition route** | Applies the 30-day rule once the form is complete |
| **Event handler** | Responds to input that matches nothing, such as text where a number is expected |
| **END_SESSION** | A special target that ends the conversation |

The finished flow looks like this:

```
Start Page --(intent: return.item)--> Collect Return Details
    form: order_id, days_since_delivery
    event handler: sys.no-match-default
    --(form complete AND days <= 30)--> Return Approved --> END_SESSION
    --(form complete AND days > 30)---> Return Declined --> END_SESSION
```

## Setup

In [ ]:
%pip install --quiet --upgrade google-cloud-dialogflow-cx

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
AGENT_DISPLAY_NAME = "cymbal-returns"
PARENT = f"projects/{PROJECT_ID}/locations/global"
print("Project:", PROJECT_ID)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Create the Agent

Every Dialogflow CX agent starts with a **Default Start Flow**, whose ID is stored in
`agent.start_flow`. If an agent with the same name already exists from an earlier run,
the cell deletes it first so that you always start from a clean state.

In [ ]:
from google.cloud import dialogflowcx_v3 as cx

agents_client = cx.AgentsClient()
for existing in agents_client.list_agents(parent=PARENT):
    if existing.display_name == AGENT_DISPLAY_NAME:
        agents_client.delete_agent(name=existing.name)
        print("Deleted the agent from a previous run.")

agent = agents_client.create_agent(
    parent=PARENT,
    agent=cx.Agent(display_name=AGENT_DISPLAY_NAME,
                   default_language_code="en",
                   time_zone="America/New_York"))
FLOW = agent.start_flow
END_SESSION = f"{FLOW}/pages/END_SESSION"
print("Created agent:", agent.name)

## Step 2: Create the Intent

An intent recognizes what the customer wants from **training phrases**. Production
agents should have at least 10 varied phrases per intent.

In [ ]:
PHRASES = [
    "I want to return my boots", "How do I send this back?", "Start a return",
    "I need a refund for an item", "This jacket doesn't fit, can I return it?",
    "Can I return something I bought?", "I'd like to return an order",
    "How do returns work?", "I want my money back for these shoes",
    "Return an item",
]

intent = cx.IntentsClient().create_intent(
    parent=agent.name,
    intent=cx.Intent(
        display_name="return.item",
        training_phrases=[
            cx.Intent.TrainingPhrase(
                parts=[cx.Intent.TrainingPhrase.Part(text=phrase)],
                repeat_count=1)
            for phrase in PHRASES]))
print("Created intent:", intent.display_name)

## Step 3: Create the Result Pages

The two result pages each send a message and then end the session through a route
with the condition `true`, which always matches. The message text uses
`$session.params.order_id` to insert a value that the form collected.

In [ ]:
pages_client = cx.PagesClient()


def say(text):
    """Builds a fulfillment that sends one text response."""
    return cx.Fulfillment(messages=[
        cx.ResponseMessage(text=cx.ResponseMessage.Text(text=[text]))])


def result_page(name, text):
    return pages_client.create_page(parent=FLOW, page=cx.Page(
        display_name=name,
        entry_fulfillment=say(text),
        transition_routes=[cx.TransitionRoute(
            condition="true", target_page=END_SESSION)]))


approved = result_page(
    "Return Approved",
    "Your return for order $session.params.order_id is approved. "
    "A return label will be emailed to you.")
declined = result_page(
    "Return Declined",
    "Order $session.params.order_id is outside the 30-day return window, "
    "so I will connect you with an agent who can review exceptions.")
print("Created pages:", approved.display_name, "and", declined.display_name)

## Step 4: Create the Form Page

This page is the heart of the flow:

- The **form** collects two required parameters, prompting for each one in order.
  `sys.number-integer` is a system entity type that accepts whole numbers.
- Two **condition routes** fire once the form is complete
  (`$page.params.status = "FINAL"`) and apply the 30-day rule.
- The **event handler** responds when the input matches nothing, such as a color where
  a number is expected.

In [ ]:
INTEGER = "projects/-/locations/-/agents/-/entityTypes/sys.number-integer"


def ask_for(name, prompt):
    return cx.Form.Parameter(
        display_name=name, required=True, entity_type=INTEGER,
        fill_behavior=cx.Form.Parameter.FillBehavior(
            initial_prompt_fulfillment=say(prompt)))


def window_routes(days):
    done = '$page.params.status = "FINAL"'
    within = f"{done} AND $session.params.days_since_delivery <= {days}"
    outside = f"{done} AND $session.params.days_since_delivery > {days}"
    return [cx.TransitionRoute(condition=within, target_page=approved.name),
            cx.TransitionRoute(condition=outside, target_page=declined.name)]


collect = pages_client.create_page(parent=FLOW, page=cx.Page(
    display_name="Collect Return Details",
    entry_fulfillment=say("I can help you start a return."),
    form=cx.Form(parameters=[
        ask_for("order_id", "What is your five-digit order number?"),
        ask_for("days_since_delivery",
                "How many days ago was the item delivered?")]),
    transition_routes=window_routes(30),
    event_handlers=[cx.EventHandler(
        event="sys.no-match-default",
        trigger_fulfillment=say("Sorry, I didn't catch that. Please reply "
                                "with numbers only, for example 10421."))]))
print("Created page:", collect.display_name)

## Step 5: Connect the Start Page and Train the Flow

The start page needs an **intent route** to the form page. The cell reads the flow,
appends the route, and saves only the `transition_routes` field. Training then
updates the flow's language model with the new intent.

In [ ]:
from google.protobuf import field_mask_pb2

flows_client = cx.FlowsClient()
start = flows_client.get_flow(name=FLOW)
start.transition_routes.append(cx.TransitionRoute(
    intent=intent.name, target_page=collect.name))
flows_client.update_flow(
    flow=start,
    update_mask=field_mask_pb2.FieldMask(paths=["transition_routes"]))

print("Training the flow...")
flows_client.train_flow(name=FLOW).result(timeout=600)
print("The flow is ready.")

## Step 6: Test the Flow

The `chat` function sends each turn of a conversation to the **DetectIntent** API in
one session and prints the agent's reply, the current page, and the parameters stored
so far.

In [ ]:
import uuid

sessions_client = cx.SessionsClient()


def chat(*turns):
    session = f"{agent.name}/sessions/{uuid.uuid4()}"
    for text in turns:
        result = sessions_client.detect_intent(
            request=cx.DetectIntentRequest(
                session=session,
                query_input=cx.QueryInput(
                    text=cx.TextInput(text=text), language_code="en"),
            )).query_result
        replies = " ".join(t for message in result.response_messages
                           for t in message.text.text)
        params = dict(result.parameters) if result.parameters else {}
        print(f"User:  {text}\nAgent: {replies}")
        print(f"       [page: {result.current_page.display_name}, "
              f"parameters: {params}]\n")

**Conversation 1: an eligible return.** Expect the agent to ask for the order number
and the number of days, move to **Return Approved**, and end the session.

In [ ]:
chat("I want to return my boots", "10421", "12")

**Conversation 2: outside the return window.** With 45 days, expect the flow to move
to **Return Declined**.

In [ ]:
chat("Start a return", "10422", "45")

**Conversation 3: unexpected input.** When the customer answers the order number
question with a description instead of a number, expect the no-match handler to ask
for numbers only, and the flow then to continue normally.

In [ ]:
chat("I want to return my boots", "the blue ones", "10423", "5")

## Step 7: Change a Business Rule

Suppose Cymbal Retail extends its return window to 45 days. Because the rule lives in
the condition routes, you can change it without touching any instructions or prompts.
The cell updates only the routes on the form page and then reruns Conversation 2.

In [ ]:
collect.transition_routes = window_routes(45)
collect = pages_client.update_page(
    page=collect,
    update_mask=field_mask_pb2.FieldMask(paths=["transition_routes"]))
print("The return window is now 45 days.\n")

chat("Start a return", "10422", "45")

**What to notice:** the same conversation now ends on **Return Approved**. Deterministic
rules are explicit, testable, and auditable, which is why the exam expects flows, not
prompts, for business rules that must never vary.

## Check Your Understanding

**1. What is the difference between a route and an event handler?**

<details><summary>Show answer</summary>

A route reacts to what the customer says (an intent) or to a condition on session data. An event handler reacts to a situation, such as input that matches nothing, no input, or a webhook error.

</details>

**2. Why do the condition routes check `$page.params.status = "FINAL"`?**

<details><summary>Show answer</summary>

So that the rule is applied only after the form has collected every required parameter.

</details>

**3. How would CX Agent Studio use a flow like this one?**

<details><summary>Show answer</summary>

It would import the flow as a flow-based agent beneath an instruction-based root agent, hand the conversation to the flow for returns, and take control back when the flow reaches END_SESSION.

</details>

## Clean Up

Run the next cell to delete the agent when you have finished the lab.

In [ ]:
agents_client.delete_agent(name=agent.name)
print("Deleted the agent", AGENT_DISPLAY_NAME)

## Summary

In this lab, you built a deterministic returns flow entirely in Python. You created an
intent, pages with entry fulfillment, a form that collects parameters, condition routes
that apply a business rule, an event handler that recovers from unexpected input, and
routes to END_SESSION. You tested the flow with DetectIntent and changed a business rule
by updating only the routes.